# ER Dashboard
Runs, metrics, hardware (CPU/GPU/throttle), and a small sample viewer. Reads only `runs/` + small samples — never full data.

In [ ]:
import json, pathlib
import pandas as pd
import matplotlib.pyplot as plt
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
RUNS = ROOT / 'runs'
pd.set_option('display.max_colwidth', 80)

## Leaderboard (all runs)

In [ ]:
lb = pd.read_csv(RUNS / 'leaderboard.csv') if (RUNS / 'leaderboard.csv').exists() else pd.DataFrame()
display(lb.sort_values('date', ascending=False) if len(lb) else 'no finished runs yet')
if len(lb):
    ax = lb.plot(x='run_id', y=['val_f05', 'val_precision', 'val_recall', 'block_recall', 'lb_score'], marker='o', figsize=(10, 4), rot=30)
    ax.set_ylim(0, 1); ax.grid(alpha=.3); plt.tight_layout()

## Current / chosen run — status, metrics, log tail

In [ ]:
RUN_ID = (RUNS / 'LATEST').read_text().strip()   # or set manually
rd = RUNS / RUN_ID
for f in ('status.json', 'metrics.json'):
    if (rd / f).exists(): print(f, json.dumps(json.loads((rd / f).read_text()), indent=1))
print(''.join((rd / 'log.txt').read_text(encoding='utf-8').splitlines(True)[-25:]))

In [ ]:
fi = rd / 'feature_importance.json'
if fi.exists():
    pd.Series(json.loads(fi.read_text())).sort_values().plot.barh(figsize=(7, 8), title='LightGBM gain'); plt.tight_layout()

## Hardware: CPU / RAM / GPU / throttling
`gpu_throttle` decodes nvidia-smi throttle reasons: `sw_power_cap`, `hw_slowdown`, `sw_thermal`, `hw_thermal`, `hw_power_brake`. Laptop GPUs often show `sw_power_cap` under load (normal); `*_thermal` = too hot.

In [ ]:
hw_path = rd / 'hw.csv'
if hw_path.exists():
    hw = pd.read_csv(hw_path, parse_dates=['time']).set_index('time')
    fig, ax = plt.subplots(4, 1, figsize=(11, 10), sharex=True)
    hw[['cpu_pct', 'ram_pct']].plot(ax=ax[0], title='CPU % / RAM %'); ax[0].set_ylim(0, 100)
    hw[['cpu_freq_mhz']].plot(ax=ax[1], title='CPU freq MHz (drops = throttling)')
    hw[['gpu_util', 'gpu_temp_c', 'gpu_power_w']].plot(ax=ax[2], title='GPU util / temp / power')
    hw[['gpu_sm_clock', 'gpu_sm_clock_max']].plot(ax=ax[3], title='GPU SM clock vs max')
    for a in ax: a.grid(alpha=.3)
    plt.tight_layout()
    display(hw['gpu_throttle'].value_counts().rename('throttle samples'))
    print('peak RAM GB', hw['ram_used_gb'].max(), '| peak proc RSS GB', hw['proc_rss_gb'].max())

## Sample viewer — a few rows only

In [ ]:
import polars as pl
CACHE = ROOT / 'cache'
p = CACHE / 'norm_train_s1.parquet'
if p.exists():
    display(pl.scan_parquet(p).head(10).collect().to_pandas())

In [ ]:
# Eyeball predictions for a few test S1 entities
out = rd / 'output' / 'matching_results.tsv'
if out.exists():
    m = pd.read_csv(out, sep='\t', nrows=2000, dtype=str).fillna('')
    print('non-empty share in first 2000:', (m.matched_entity_ids != '').mean())
    display(m[m.matched_entity_ids != ''].head(10))